# Signal Research Notebook

Interactive research and development notebook for signal creation and testing.

## Workflow
1. **Load & Explore Data** - Load market data and inspect distributions
2. **Develop Signal** - Build and test signal logic interactively
3. **Implement** - Copy validated logic to `signal.py`
4. **Execute** - Run `uv run create-signal` to generate `data/signal.parquet`
5. **Validate Signal** - use `uv run ew-dash` to view signal characteristics.
6. **Backtest** - Use `uv run backtest` for slurm backtest on super computer
7. **Performance** - Use `uv run opt-dash` for in depth analysis of mvo backtested signal.

## Tips
- Use cells to isolate different aspects of your signal
- Modify parameters directly in cells to test variations
- Check signal statistics regularly to catch issues early
- Document your assumptions and findings as you develop

## Setup

In [1]:
import polars as pl
import numpy as np
import datetime as dt
import sf_quant.data as sfd
import sf_quant.research as sfr
import polars_ols

In [2]:
import os
import polars as pl
import datetime as dt
from dotenv import load_dotenv
import sf_quant.data as sfd

## 1. Load & Explore Data

Load your market data and inspect key characteristics before developing the signal.

In [3]:
start = dt.date(2000, 1, 1)
end = dt.date(2024, 12, 31)

data = sfd.load_assets(
        start=start,
        end=end,
        columns=[
            "date",
            "barrid",
            "price",
            "return",
            "specific_risk",
            "predicted_beta",
            "daily_volume",
            "market_cap",
        ],
        in_universe=True,
        ).with_columns(pl.col("return", "specific_risk").truediv(100))

data

date,barrid,price,return,specific_risk,predicted_beta,daily_volume,market_cap
date,str,f64,f64,f64,f64,f64,f64
2013-07-31,"""USA06Z1""",6.26,-0.001595,0.550569,0.34349,121693.0,6.006157e8
2013-08-01,"""USA06Z1""",6.32,0.009585,0.55028,0.353329,131728.0,6.0865392e8
2013-08-02,"""USA06Z1""",6.31,-0.001582,0.548074,0.363624,43252.0,6.0769086e8
2013-08-05,"""USA06Z1""",6.45,0.022187,0.547667,0.356596,70944.0,6.211737e8
2013-08-06,"""USA06Z1""",6.29,-0.024806,0.546922,0.399196,77085.0,6.0576474e8
…,…,…,…,…,…,…,…
2024-12-24,"""USBQOR1""",70.58,0.025872,0.268004,1.287294,47177.0,3.5976e9
2024-12-26,"""USBQOR1""",73.61,0.04293,0.271723,1.288943,69860.0,3.7521e9
2024-12-27,"""USBQOR1""",69.85,-0.05108,0.274681,1.294801,98307.0,3.5604e9


In [5]:
def load_data() -> pl.DataFrame:
    """
    Load and prepare market data for signal creation.

    Returns:
        pl.DataFrame: Market data with required columns
    """
    # TODO: Load data from source (API, file, database)
    start = dt.date(1927, 1, 1)
    end = dt.date(2024, 12, 31)

    data = sfd.load_assets(
        start=start,
        end=end,
        columns=[
            "date",
            "barrid",
            "price",
            "return",
            "specific_risk",
            "predicted_beta",
            "daily_volume",
            "market_cap",
        ],
        in_universe=True,
        ).with_columns(pl.col("return", "specific_risk").truediv(100))

    # TODO: Filter data as needed (date range, symbols, quality checks)

    return data


def create_signal():
    """
    Loads data, creates a simple signal, and saves it to parquet.
    """
    # Load environment variables from .env file
    load_dotenv()
    project_root = os.getcwd()
    output_path = os.getenv("SIGNAL_PATH", "data/signal.parquet")
    if not os.path.isabs(output_path):
        output_path = os.path.join(project_root, output_path)
    os.makedirs(os.path.dirname(output_path), exist_ok=True)


    # TODO: Load Data
    df = load_data()

    # TODO: Add your signal logic here (remember alpha logic)
    #init turnover
    df = df.sort(["barrid", "date"])
    df = df.with_columns(
        (pl.col("daily_volume") / pl.col("market_cap")).alias("turnover")
    )

    #init turnover mean and std
    df = df.with_columns(
        pl.col("turnover").rolling_mean(252).over("barrid").alias("turnover_mean"),
        pl.col("turnover").rolling_std(252).over("barrid").alias("turnover_std"),
    )

    #calc satv signal
    df = df.with_columns(
        ((pl.col("turnover") - pl.col("turnover_mean")) / pl.col("turnover_std"))
        .shift(1)
        .over("barrid")
        .alias("signal")
    )

    #some filters
    df = df.filter(
        (pl.col("signal").is_not_null()) &
        (pl.col("price") >= 5)
    )

    #ic
    IC = 0.05

    #compute z scores for alpha
    scores = df.select(
        "date",
        "barrid",
        "predicted_beta",
        "specific_risk",
        "signal",
        "return",
        pl.col('signal')
        .sub(pl.col('signal').mean())
        .truediv(pl.col('signal').std())
        .over("date")
        .alias("score"),
    )

    #compute alphas
    alphas = (
        scores.with_columns(pl.col("score").mul(IC).mul("specific_risk").alias("alpha"))
        # .select("date", "barrid", "alpha", "signal", "return", "predicted_beta")
        .sort("date", "barrid")
    )

    alphas.write_parquet(output_path)
    return alphas

In [6]:
df = create_signal()
df.select(pl.corr("signal", "return"))

signal
f64
0.016117


## 2. Signal Development

Build and test your signal logic. Modify parameters and logic here to find optimal configurations.

In [ ]:
def create_signal(df: pl.DataFrame) -> pl.DataFrame:
    """
    Create signal based on market data.
    
    Args:
        df: Market data DataFrame
        
    Returns:
        pl.DataFrame: DataFrame with signal column added
    """
    # TODO: Implement signal calculation, include alpha logic.
    return df

signal = create_signal(df)
signal.head()

## 3. Signal Analysis

Examine signal statistics and distributions to understand its characteristics.

### Statistics

In [ ]:
# sfr.get_signal_stats(signal)

### Distribution

In [ ]:
# sfr.get_signal_distribution(signal)

## 4. Validation Checks ?

Verify signal quality and identify any issues before implementation.

## 5. Next Steps

When satisfied with your signal:

1. **Copy** your data loading and signal calculation logic to `signal.py`
2. **Run** `uv run create-signal` to save the signal to `data/signal.parquet`
3. **Open** `uv run ew-dash` to analyze the signal before backtesting